# 02 — Full train + Hugging Face publish + side-by-side demo

Kaggle **GPU (T4)** notebook that:

1. Builds the grounded dataset (Phase 1)
2. Runs **full-epoch** Unsloth QLoRA SFT (Phase 2)
3. Runs a **side-by-side base vs fine-tuned** research-panel compare
4. Uploads the LoRA adapter to the **Hugging Face Hub**
5. Optionally launches a **Gradio** share demo with base | after columns

| Flag | Default | Meaning |
|------|---------|---------|
| `RUN_TRAIN` | `True` | Actually train (needs T4) |
| `MAX_STEPS` | `None` | Full epoch from YAML |
| `DOWNLOAD_HF` | `True` | Stream small public HF samples |
| `RUN_SIDE_BY_SIDE` | `True` | Base vs adapter panel compare after train |
| `SIDE_BY_SIDE_LIMIT` | `4` | How many panel items to compare (VRAM time) |
| `PUBLISH_HF` | `True` | Upload adapter after train |
| `LAUNCH_GRADIO` | `False` | Share Gradio link (blocks until stopped) |
| `GRADIO_SIDE_BY_SIDE` | `True` | Gradio shows base | fine-tuned columns |

**Secrets (Kaggle → Add-ons → Secrets):** add `HF_TOKEN` with a **write** token. Never paste the token into a cell.

Adapter path: `outputs/adapters/llama32-3b-ecra-sft/`  
Default Hub repo: `nuwanda94/llama32-3b-ecra-sft` (change `HF_REPO_ID` below).

## 0. Knobs

In [ ]:
# --- User knobs ---
RUN_TRAIN = True
MAX_STEPS = None                 # None = full epoch (num_train_epochs in YAML)
MAX_SAMPLES = 50
DOWNLOAD_HF = True
USE_LLM_JUDGE = False
CONFIG_PATH = "configs/default.yaml"

# Side-by-side base vs fine-tuned (after train)
RUN_SIDE_BY_SIDE = True
SIDE_BY_SIDE_LIMIT = 4           # panel items; sequential loads to fit T4

# Hugging Face Hub
PUBLISH_HF = True
HF_REPO_ID = "nuwanda94/llama32-3b-ecra-sft"
HF_PRIVATE = False

# Gradio (share=True gives a temporary public URL)
LAUNCH_GRADIO = False
GRADIO_SHARE = True
GRADIO_SIDE_BY_SIDE = True       # base | fine-tuned columns

ADAPTER_DIR = "outputs/adapters/llama32-3b-ecra-sft"

print("RUN_TRAIN", RUN_TRAIN, "MAX_STEPS", MAX_STEPS)
print("RUN_SIDE_BY_SIDE", RUN_SIDE_BY_SIDE, "limit", SIDE_BY_SIDE_LIMIT)
print("PUBLISH_HF", PUBLISH_HF, "repo", HF_REPO_ID)
print("LAUNCH_GRADIO", LAUNCH_GRADIO, "side_by_side", GRADIO_SIDE_BY_SIDE)

## 1. Repo path + installs

In [ ]:
from pathlib import Path
import os
import sys

IN_KAGGLE = Path("/kaggle").exists()
print("IN_KAGGLE:", IN_KAGGLE)

if IN_KAGGLE:
    work = Path("/kaggle/working")
    repo = work / "earnings-call-research-assistant"
    if not (repo / "src" / "earnings_call_research_assistant" / "inference.py").exists():
        %cd /kaggle/working
        !rm -rf earnings-call-research-assistant
        !git clone --depth 1 https://github.com/nuwanda94/earnings-call-research-assistant.git
        repo = work / "earnings-call-research-assistant"
    REPO = repo.resolve()
else:
    REPO = Path("..").resolve()
    if not (REPO / "src" / "earnings_call_research_assistant").is_dir():
        REPO = Path.cwd().resolve()

SRC = REPO / "src"
assert (SRC / "earnings_call_research_assistant" / "inference.py").exists(), SRC
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
print("REPO:", REPO)
print("cwd:", Path.cwd())

import earnings_call_research_assistant as ecra
print("package:", ecra.__file__, "v", getattr(ecra, "__version__", "?"))

In [ ]:
if IN_KAGGLE:
    %pip install -q pyyaml huggingface_hub datasets
    if RUN_TRAIN or RUN_SIDE_BY_SIDE or LAUNCH_GRADIO:
        %pip install -q unsloth transformers accelerate bitsandbytes trl peft
    if LAUNCH_GRADIO:
        %pip install -q gradio

## 2. Wire Hugging Face token from Kaggle secrets

Create a secret named **`HF_TOKEN`** (write access). This cell never prints the token value.

In [ ]:
def _load_hf_token() -> bool:
    # 1) Already in env
    if os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"):
        return True
    # 2) Kaggle secrets
    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
            if tok and tok.strip():
                os.environ["HF_TOKEN"] = tok.strip()
                return True
        except Exception as e:
            print("Kaggle secrets note:", type(e).__name__, str(e)[:120])
    return False

has_token = _load_hf_token()
print("HF token present:", has_token, "(value not shown)")
if PUBLISH_HF and not has_token:
    print("WARNING: PUBLISH_HF=True but no token. Add Kaggle secret HF_TOKEN or export HF_TOKEN.")

## 3. Phase 1 — data pipeline

In [ ]:
from earnings_call_research_assistant.data import (
    DATASET_VERSION,
    ChunkConfig,
    FilterConfig,
    GenerateConfig,
    SelectConfig,
    chunk_records,
    filter_pairs,
    generate_pairs,
    ingest_catalog,
    list_sources,
    select_and_split,
    write_chunks_jsonl,
    write_filter_report,
    write_jsonl,
    write_pairs_jsonl,
    write_splits,
)

print("Sources:")
for s in list_sources():
    print(f"  - {s.source_id}: {s.display_name}")

records = ingest_catalog(max_samples=MAX_SAMPLES, download=DOWNLOAD_HF)
write_jsonl(records, Path("data/raw/public_sample.jsonl"))
print("records:", len(records))

chunks = chunk_records(records, config=ChunkConfig(window_sentences=4, stride_sentences=2))
write_chunks_jsonl(chunks, Path("data/processed/chunks.jsonl"))
print("chunks:", len(chunks), "props:", sum(len(c.propositions) for c in chunks))

pairs = generate_pairs(
    chunks, config=GenerateConfig(max_qa_per_chunk=2, include_summary=True, use_llm=False)
)
write_pairs_jsonl(pairs, Path("data/processed/grounded_pairs.jsonl"))
print("pairs:", len(pairs))

kept, report = filter_pairs(
    pairs,
    config=FilterConfig(
        min_output_chars=40,
        near_dup_jaccard=0.88,
        use_llm_judge=USE_LLM_JUDGE,
        min_judge_score=0.6,
    ),
)
write_pairs_jsonl(kept, Path("data/processed/filtered_pairs.jsonl"))
write_filter_report(report, Path("data/processed/filter_report.json"))
print("kept:", report.n_kept, "dropped_by_stage:", report.dropped_by_stage)

OUT_DIR = Path("data/processed") / DATASET_VERSION
sel_cfg = SelectConfig(
    target_min=1,
    target_max=6000,
    max_per_source=2500,
    diversity_jaccard_cap=0.72,
    seed=94,
    dataset_version=DATASET_VERSION,
)
splits, sel_report = select_and_split(kept, config=sel_cfg)
paths = write_splits(splits, OUT_DIR, report=sel_report, config=sel_cfg)
print(
    f"selected={sel_report.n_selected} train={sel_report.n_train} "
    f"val={sel_report.n_val} test={sel_report.n_test}"
)
print("train path:", paths["train"])

## 4. SFT dry-run plan

In [ ]:
from earnings_call_research_assistant.training.sft import run_sft
import json

plan = run_sft(
    config_path=CONFIG_PATH,
    dataset_dir=OUT_DIR,
    dry_run=True,
    max_steps=MAX_STEPS,
    require_train=False,
)
print(
    f"dry_run={plan.dry_run} model={plan.model_name} seed={plan.seed} "
    f"train={plan.n_train} val={plan.n_val}"
)
print("adapter_dir:", plan.adapter_dir)
print(json.dumps({k: plan.to_dict()[k] for k in ("dry_run", "n_train", "n_val", "seed", "model_name", "adapter_dir")}, indent=2))

## 5. Full QLoRA train

Requires **T4 GPU**. With `MAX_STEPS=None` this runs `num_train_epochs` from the YAML (default 1).

In [ ]:
import torch

print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

if not RUN_TRAIN:
    print("Skipped train (RUN_TRAIN=False).")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a T4 GPU in Kaggle notebook settings.")
    if plan.n_train < 1:
        raise RuntimeError("Empty train split — fix data cells first.")
    train_plan = run_sft(
        config_path=CONFIG_PATH,
        dataset_dir=OUT_DIR,
        dry_run=False,
        max_steps=MAX_STEPS,
        require_train=True,
    )
    print("Train finished.")
    print("adapter:", train_plan.adapter_dir)
    print("notes:", train_plan.notes[-3:])

adapter_path = Path(ADAPTER_DIR)
print("adapter exists:", adapter_path.exists())
if adapter_path.exists():
    print("files:", sorted(p.name for p in adapter_path.iterdir())[:15])

## 6. Side-by-side base vs fine-tuned (research panel)

Loads **base**, generates answers, frees GPU, then loads the **adapter** and generates the same prompts. Writes `evals/reports/side_by_side_panel.jsonl` for portfolio evidence.

Requires GPU + an adapter on disk (`RUN_TRAIN` or a pre-existing `ADAPTER_DIR`).

In [ ]:
import gc
import json
from earnings_call_research_assistant.eval.panel import load_panel, DEFAULT_PANEL
from earnings_call_research_assistant.inference import InferenceConfig, InferenceHarness
import yaml

def _release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _load_cfg():
    with Path(CONFIG_PATH).open() as f:
        return InferenceConfig.from_mapping(yaml.safe_load(f))

def _generate_batch(harness, items):
    rows = []
    for item in items:
        text = item.user_text()
        reply = harness.generate(text)
        rows.append({"id": item.id, "ticker": item.ticker, "theme": item.theme,
                     "user_text": text, "reply": reply})
        print(f"  [{item.id}] {item.theme[:40]}… -> {len(reply)} chars")
    return rows

compare_path = Path("evals/reports/side_by_side_panel.jsonl")
compare_path.parent.mkdir(parents=True, exist_ok=True)

if not RUN_SIDE_BY_SIDE:
    print("Skipped side-by-side (RUN_SIDE_BY_SIDE=False).")
elif not torch.cuda.is_available():
    print("Skipped side-by-side: no CUDA.")
elif not Path(ADAPTER_DIR).exists():
    print(f"Skipped side-by-side: no adapter at {ADAPTER_DIR}. Train first.")
else:
    panel = load_panel(DEFAULT_PANEL)[: max(1, int(SIDE_BY_SIDE_LIMIT))]
    print(f"Comparing {len(panel)} panel items…")

    cfg = _load_cfg()
    print("Loading BASE…")
    base_h = InferenceHarness.from_pretrained(cfg)
    base_rows = _generate_batch(base_h, panel)
    del base_h
    _release()

    print("Loading ADAPTER…", ADAPTER_DIR)
    try:
        tuned_h = InferenceHarness.from_pretrained(cfg, model_name=ADAPTER_DIR)
    except Exception as e:
        print("Direct adapter load failed, base + load_adapter:", e)
        tuned_h = InferenceHarness.from_pretrained(cfg)
        tuned_h.model.load_adapter(ADAPTER_DIR)
    tuned_rows = _generate_batch(tuned_h, panel)
    del tuned_h
    _release()

    by_id = {r["id"]: r for r in tuned_rows}
    merged = []
    for br in base_rows:
        tr = by_id.get(br["id"], {})
        merged.append({
            "id": br["id"],
            "ticker": br["ticker"],
            "theme": br["theme"],
            "user_text": br["user_text"],
            "base_reply": br["reply"],
            "adapter_reply": tr.get("reply", ""),
        })

    with compare_path.open("w", encoding="utf-8") as f:
        for row in merged:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("Wrote", compare_path)

    print("\n=== SIDE-BY-SIDE ===")
    for row in merged:
        print("\n---", row["id"], row["ticker"], row["theme"], "---")
        print("[BASE]\n", row["base_reply"][:500])
        print("[ADAPTER]\n", row["adapter_reply"][:500])

## 7. Smoke-generate with adapter

In [ ]:
if not Path(ADAPTER_DIR).exists():
    print("No adapter on disk — skip smoke generate.")
else:
    with Path(CONFIG_PATH).open() as f:
        cfg = InferenceConfig.from_mapping(yaml.safe_load(f))
    try:
        harness = InferenceHarness.from_pretrained(cfg, model_name=ADAPTER_DIR)
        print("Loaded from adapter dir:", ADAPTER_DIR)
    except Exception as e:
        print("Direct adapter load failed, base + load_adapter:", e)
        harness = InferenceHarness.from_pretrained(cfg)
        try:
            harness.model.load_adapter(ADAPTER_DIR)
        except Exception as e2:
            print("load_adapter failed:", e2)
    print(harness.generate(
        "Summarize prepared remarks vs Q&A on a US large-cap quarterly earnings call."
    ))
    del harness
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 8. Publish adapter to Hugging Face Hub

Uses `earnings_call_research_assistant.publish.publish_adapter`. Token from env / Kaggle secrets only.

In [ ]:
from earnings_call_research_assistant.publish import publish_adapter

# Always write a dry-run plan first
dry = publish_adapter(
    adapter_dir=ADAPTER_DIR,
    repo_id=HF_REPO_ID,
    private=HF_PRIVATE,
    dry_run=True,
)
print("dry plan: exists=", dry.adapter_exists, "looks_like=", dry.looks_like_adapter,
      "token=", dry.token_present)
print("notes:", dry.notes)

if not PUBLISH_HF:
    print("Skipped upload (PUBLISH_HF=False).")
elif not dry.adapter_exists or not dry.looks_like_adapter:
    raise FileNotFoundError(f"Cannot publish: bad adapter dir {ADAPTER_DIR}")
elif not dry.token_present:
    raise RuntimeError("Cannot publish: set HF_TOKEN (Kaggle secret or env).")
else:
    live = publish_adapter(
        adapter_dir=ADAPTER_DIR,
        repo_id=HF_REPO_ID,
        private=HF_PRIVATE,
        commit_message="feat: upload ECRA QLoRA adapter from Kaggle full train",
        dry_run=False,
    )
    print("uploaded:", live.uploaded)
    print("hub_url:", live.hub_url)
    print("Open:", live.hub_url or f"https://huggingface.co/{HF_REPO_ID}")

## 9. Gradio demo — side-by-side base | fine-tuned (optional)

Set `LAUNCH_GRADIO = True` to start a shareable Gradio app. With `GRADIO_SIDE_BY_SIDE = True` the UI shows **Base (before)** and **Fine-tuned (after)** columns. This cell **blocks** until you stop it.

Each click loads base then adapter **sequentially** so a single T4 can run the compare (slower, but no dual-model VRAM).

In [ ]:
if not LAUNCH_GRADIO:
    print("Skipped Gradio (LAUNCH_GRADIO=False). Set True and re-run after train.")
else:
    from earnings_call_research_assistant.demo import launch_demo
    adapter = ADAPTER_DIR if Path(ADAPTER_DIR).exists() else None
    print(
        "Launching Gradio adapter=", adapter,
        "share=", GRADIO_SHARE,
        "side_by_side=", GRADIO_SIDE_BY_SIDE,
    )
    launch_demo(
        load_model=True,
        config_path=CONFIG_PATH,
        adapter_dir=adapter,
        share=GRADIO_SHARE,
        server_name="0.0.0.0",
        side_by_side=GRADIO_SIDE_BY_SIDE,
    )

## Done

1. Side-by-side JSON: `evals/reports/side_by_side_panel.jsonl`
2. Confirm Hub page: `https://huggingface.co/<HF_REPO_ID>`
3. Model card tip: set base model to `unsloth/Llama-3.2-3B-Instruct` and note QLoRA + seed `3407`
4. Local side-by-side Gradio after publish/train:

```bash
python scripts/demo_gradio.py --run --side-by-side \
  --adapter-dir outputs/adapters/llama32-3b-ecra-sft --share
```

5. Full panel eval on GPU: `python scripts/eval_research_panel.py --run --adapter-dir outputs/adapters/llama32-3b-ecra-sft`

See `docs/REPRODUCIBILITY.md` and `docs/DATA_CARD.md`.